<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/seq2one/stage_07_02b_mlp_seq2one_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_02b - SEQ2ONE - MLP - Tuning**



# **BLOQUE DE EJECUCIÓN COMPLETO**

In [48]:
window_sizes = [180]
targets = ['delta_60']
splits = ['train', 'valid', 'test']

## **1. Imports + paths**

In [49]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [50]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Rutas de ventanas seq2one y scalers**

In [51]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

In [52]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [53]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl')}

## **4. Reproducibilidad**

In [54]:
#def set_seeds(seed: int = 42) -> None:
#    random.seed(seed)
#    np.random.seed(seed)
#    os.environ["PYTHONHASHSEED"] = str(seed)
#
#set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [55]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [56]:
#print(compute_seq2one_metrics.__doc__)

## **6. Carga de data windows**

In [57]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [58]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [59]:
from typing import Any, Dict, Mapping
from pathlib import Path

# --------------------------------------------------
# Carga completa: ventanas + scaler por window_size y target
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path
    scalers_path[target] -> Path
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]
    scaler_path = scalers_paths[target]

    # --------------------------
    # 3) Carga
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    scaler = load_scaler(scaler_path)

    # --------------------------
    # 4) Inferir horizonte
    # --------------------------
    horizon = int(target.split("_")[-1])

    # --------------------------
    # 5) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [60]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [61]:
def create_bundles(window_size, targets: list, windows_paths=windows_paths, scalers_paths=scalers_paths, *, flatten_X=False):

    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
        )
        if flatten_X:
            b["train"]["X"] = maybe_flatten_X(b["train"]["X"], flatten=True)
            b["valid"]["X"] = maybe_flatten_X(b["valid"]["X"], flatten=True)
            b["test"]["X"]  = maybe_flatten_X(b["test"]["X"],  flatten=True)
        bundles.append(b)

    # prints (opcional)
    for b in bundles:
        print(f"H{b['horizon']} Train:", b["train"]["X"].shape, b["train"]["y"].shape)
        print(f"H{b['horizon']} Valid:", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print(f"H{b['horizon']} Test :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [62]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [63]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [64]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [65]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML**

In [66]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [67]:
def get_metrics_torch(bundle, model, *, device) -> tuple[dict, dict]:

    # -------- VALID --------
    X_valid = bundle["valid"]["X"]
    y_valid = np.asarray(bundle["valid"]["y"]).reshape(-1)
    y_pred_valid = np.asarray(
        predict_mlp(model, X_valid, device=device)
    ).reshape(-1)

    metrics_valid = compute_seq2one_metrics(
        y_valid, y_pred_valid, compute_r2=True
    )

    # -------- TEST --------
    X_test = bundle["test"]["X"]
    y_test = np.asarray(bundle["test"]["y"]).reshape(-1)
    y_pred_test = np.asarray(
        predict_mlp(model, X_test, device=device)
    ).reshape(-1)

    metrics_test = compute_seq2one_metrics(
        y_test, y_pred_test, compute_r2=True
    )

    return metrics_valid, metrics_test

## **9. Gestión de dataset de métricas**

In [68]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [69]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [70]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
torch.cuda.empty_cache()

# **DEFINICIÓN DE MODELO**

## **10. Definición del modelo — placeholder**

### **10.1. Baseline MLP Configuration (SEQ2ONE – delta_60)**


**Hiperparámetros baseline**


| Hiperparámetro   | Valor   |
|------------------|---------|
| in_dim           | 1200    |
| hidden_dim       | 128     |
| dropout          | 0.0     |
| learning_rate    | 1e-3    |
| weight_decay     | 1e-4    |
| max_epochs       | 30      |
| patience         | 5       |
| optimizer        | Adam    |
| loss_function    | MSE     |

In [71]:
import pandas as pd
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

df_mlp_all_sizes = load_seq2one_metrics_if_exists(name="mlp")
df_mlp_delta_60 = df_mlp_all_sizes[df_mlp_all_sizes["target"] == "delta_60"].copy()

In [72]:
df_mlp_delta_60

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,hidden_dim,dropout,lr,w_decay,batch_size,max_epochs,patience
0,mlp,test,30,delta_60,60,48.197694,75.094125,0.158334,0.612614,128,0.0,0.001,0.0001,16384,30,5
1,mlp,valid,30,delta_60,60,30.792758,44.250947,0.189710,0.635432,128,0.0,0.001,0.0001,16384,30,5
8,mlp,test,60,delta_60,60,41.986391,67.616881,0.323348,0.696595,128,0.0,0.001,0.0001,16384,30,5
9,mlp,valid,60,delta_60,60,25.739746,39.123824,0.370360,0.715509,128,0.0,0.001,0.0001,16384,30,5
16,mlp,test,90,delta_60,60,39.345348,65.865070,0.354628,0.704880,128,0.0,0.001,0.0001,16384,30,5
17,mlp,valid,90,delta_60,60,23.108619,37.027374,0.422718,0.738037,128,0.0,0.001,0.0001,16384,30,5
24,mlp,test,120,delta_60,60,38.710646,64.403748,0.380758,0.709230,128,0.0,0.001,0.0001,16384,30,5
25,mlp,valid,120,delta_60,60,23.325614,37.396104,0.395885,0.733588,128,0.0,0.001,0.0001,16384,30,5
32,mlp,test,180,delta_60,60,36.526562,61.421779,0.447396,0.731242,128,0.0,0.001,0.0001,16384,30,5
33,mlp,valid,180,delta_60,60,21.795650,35.331207,0.450572,0.744949,128,0.0,0.001,0.0001,16384,30,5


### **11.2. Imports (PyTorch) + semillas**

In [73]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [74]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # reproducibilidad (puede bajar performance, pero estable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


### **11.2. Dataset/DataLoader desde bundle**

In [75]:
def make_loaders_from_bundle(bundle, *, batch_size: int = 4096, num_workers: int = 0) -> dict:
    loaders = {}
    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 2:
            raise ValueError(f"[{split}] X debe ser 2D (n, d). Got shape={X.shape}")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"[{split}] X y y deben tener mismo n. X={X.shape}, y={y.shape}")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )
    return loaders

In [76]:
#loaders_60 = make_loaders_from_bundle(bundle_60, batch_size=16384)
#loaders_90 = make_loaders_from_bundle(bundle_90, batch_size=16384)

### **11.3. Definición del modelo MLP (simple y controlado)**

In [77]:
import torch
import torch.nn as nn

def _get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    if name == "silu" or name == "swish":
        return nn.SiLU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"activation no soportada: {name}")

class MLPSeq2One(nn.Module):
    """
    MLP configurable para seq2one:
      - in_dim: dimensión de entrada (d)
      - hidden_dims: lista de dimensiones ocultas, p.ej. [256, 256, 128]
      - activation: relu | gelu | silu | tanh
      - dropout: float
      - norm: none | layernorm | batchnorm
      - residual: aplica skip en bloques con misma dim (opcional)
    """
    def __init__(
        self,
        *,
        in_dim: int,
        hidden_dims: list[int],
        activation: str = "relu",
        dropout: float = 0.0,
        norm: str = "none",
        residual: bool = False,
    ):
        super().__init__()
        if not hidden_dims:
            raise ValueError("hidden_dims debe tener al menos 1 capa")

        act = _get_activation(activation)
        norm = norm.lower()

        dims = [in_dim] + list(hidden_dims)
        layers: list[nn.Module] = []

        for i in range(len(dims) - 1):
            d_in, d_out = dims[i], dims[i + 1]

            block: list[nn.Module] = [nn.Linear(d_in, d_out)]

            if norm == "layernorm":
                block.append(nn.LayerNorm(d_out))
            elif norm == "batchnorm":
                block.append(nn.BatchNorm1d(d_out))
            elif norm == "none":
                pass
            else:
                raise ValueError(f"norm no soportada: {norm}")

            block.append(act)
            if dropout and dropout > 0:
                block.append(nn.Dropout(dropout))

            layers.append(nn.Sequential(*block))

        self.blocks = nn.ModuleList(layers)
        self.residual = residual
        self.out = nn.Linear(dims[-1], 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # asegurar 2D: (n, d)
        if x.ndim != 2:
            x = x.view(x.size(0), -1)

        h = x
        for block in self.blocks:
            h_new = block(h)
            # residual sólo si misma dimensión
            if self.residual and (h_new.shape[-1] == h.shape[-1]):
                h = h + h_new
            else:
                h = h_new

        return self.out(h)

### **11.4. Entrenamiento con early stopping (VALID)**

In [78]:
@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    sse = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        sse += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return sse / max(n, 1)

def train_mlp(
    loaders: dict,
    *,
    model_cfg: dict,      # <-- arquitectura
    optim_cfg: dict,      # <-- hiperparams aprendizaje
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,
):
    """
    Entrena MLP seq2one con early stopping en VALID por MSE.
    Retorna: (model_best, best_valid_mse, epochs_ran)
    """
    if verbose:
        print(
            f"[{_ts()}] [TRAIN] START | model_cfg={model_cfg} | "
            f"optim_cfg={optim_cfg} | max_epochs={max_epochs} patience={patience} | device={device.type}"
        )

    t_global = time.perf_counter()

    model = MLPSeq2One(**model_cfg).to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=float(optim_cfg.get("lr", 1e-3)),
        weight_decay=float(optim_cfg.get("weight_decay", 0.0)),
    )
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0
    epochs_ran = 0

    for epoch in range(1, max_epochs + 1):
        epochs_ran = epoch
        t_epoch = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={train_loss_sum/max(train_n,1):.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        t0 = time.perf_counter()
        valid_mse = evaluate_mse(model, loaders["valid"], device)
        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        improved = valid_mse < (best_valid - 1e-9)
        if improved:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(
                f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | best_valid_mse={best_valid:.6f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} | epochs_ran={epochs_ran} | dt_total={dt_all:.2f}s")

    return model, best_valid, epochs_ran

### **11.5. Predicciones MLP**


In [79]:
@torch.no_grad()
def predict_mlp(model, X: np.ndarray, *, device: torch.device, batch_size: int = 32768) -> np.ndarray:
    """
    Predice con un modelo MLP PyTorch en batches.
    Retorna shape (n_samples,)
    """
    model.eval()

    X = np.asarray(X, dtype=np.float32)

    # 🔎 Validación crítica de shape
    if X.ndim != 2:
        raise ValueError(f"X debe ser 2D (n, d). Got shape={X.shape}")

    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb).squeeze(-1)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)

## **12. Ejecución completa**

In [80]:
def run_mlp(
    window_size: int,
    *,
    seed: int,
    model_cfg: dict,
    optim_cfg: dict,
    train_cfg: dict,
    verbose: bool = True,
):
    set_seed(seed)

    size = window_size
    n_features = int(train_cfg.get("n_features", 36))

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"[{_ts()}] MLP | SEQ2ONE | seed={seed} | L{size} | in_dim={size*n_features}\n"
            f"model_cfg={model_cfg}\noptim_cfg={optim_cfg}\ntrain_cfg={train_cfg}"
        )
        print("=" * 80)

    targets = train_cfg.get("targets", ["delta_60", "delta_90", "ret_60", "ret_90"])
    rows = []
    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        t_target = time.perf_counter()
        if verbose:
            print(f"\n[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{size}")

        # BUILD BUNDLE
        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,
        )

        # LOADERS
        loaders = make_loaders_from_bundle(
            bundle,
            batch_size=int(train_cfg.get("batch_size", 16384)),
            num_workers=int(train_cfg.get("num_workers", 0)),
        )

        # TRAIN
        _model_cfg = dict(model_cfg)
        _model_cfg["in_dim"] = size * n_features  # forzar consistencia con L y n_features

        model, best_valid_mse, epochs_ran = train_mlp(
            loaders,
            model_cfg=_model_cfg,
            optim_cfg=optim_cfg,
            max_epochs=int(train_cfg.get("max_epochs", 30)),
            patience=int(train_cfg.get("patience", 5)),
            device=device,
            verbose=verbose,
            log_every=int(train_cfg.get("log_every", 0)),
        )

        # liberar TRAIN
        del bundle["train"]
        gc.collect()

        # METRICS
        metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)

        # DF APPEND (+ registrar seed y best_valid_mse)
        df_va = metrics_to_df(
            metrics_valid, model="mlp", split="valid",
            horizon=bundle["horizon"], window_size=bundle["window_size"], target=bundle["target"],
        )
        df_te = metrics_to_df(
            metrics_test, model="mlp", split="test",
            horizon=bundle["horizon"], window_size=bundle["window_size"], target=bundle["target"],
        )

        for df_ in (df_va, df_te):
            df_["seed"] = seed
            df_["best_valid_mse"] = best_valid_mse
            df_["epochs_ran"] = epochs_ran
            # opcional: guardar config serializada
            df_["model_cfg"] = str(_model_cfg)
            df_["optim_cfg"] = str(optim_cfg)

        rows.append(df_va)
        rows.append(df_te)

        # CLEANUP
        del bundle, model, metrics_valid, metrics_test, loaders, df_va, df_te
        gc.collect()

        if verbose:
            print(f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | dt_total={time.perf_counter()-t_target:.2f}s")

    df_mlp_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model", "seed"])
          .reset_index(drop=True)
    )

    if verbose:
        print(f"\n[{_ts()}] [FINAL] OK | rows={len(df_mlp_metrics)} | dt_total={time.perf_counter()-t_global:.2f}s")

    return df_mlp_metrics

In [81]:
import pandas as pd
import gc

def run_mlp_incremental(
    window_sizes: list[int],
    *,
    seeds: list[int],
    configs: list[dict],   # cada item: {"cfg_id": "...", "model_cfg": {...}, "optim_cfg": {...}, "train_cfg": {...}}
    name: str = "mlp",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    if df_all.empty:
        df_all = pd.DataFrame(columns=[
            "cfg_id","seed","model","split","window_size","target","horizon_min",
            "MAE","RMSE","R2","DA",
            "best_valid_mse","epochs_ran",
        ])

    # llaves únicas por corrida
    key_cols = ["cfg_id","seed","model","window_size","target","split","horizon_min"]

    for ws in window_sizes:
        for cfg in configs:
            cfg_id = cfg["cfg_id"]

            for seed in seeds:
                # verificar si ya existe TODO para este (ws, cfg_id, seed)
                df_exist = df_all[
                    (df_all["cfg_id"] == cfg_id) &
                    (df_all["seed"] == seed) &
                    (df_all["window_size"] == ws) &
                    (df_all["model"] == "mlp")
                ]

                # esperado: 4 targets x 2 splits = 8 filas POR seed y cfg
                if len(df_exist) >= 8:
                    if verbose:
                        print(f"[SKIP] L{ws} cfg={cfg_id} seed={seed}: ya hay {len(df_exist)} filas.")
                    continue

                if verbose:
                    print("\n" + "="*90)
                    print(f"[RUN] MLP incremental | L{ws} | cfg={cfg_id} | seed={seed}")
                    print("="*90)

                df_new = run_mlp(
                    ws,
                    seed=seed,
                    model_cfg=cfg["model_cfg"],
                    optim_cfg=cfg["optim_cfg"],
                    train_cfg=cfg["train_cfg"],
                    verbose=verbose,
                ).copy()

                df_new["cfg_id"] = cfg_id
                df_new["seed"] = seed

                # anti-duplicados
                existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
                mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
                df_new = df_new.loc[mask_keep].copy()

                if df_new.empty:
                    if verbose:
                        print(f"[INFO] L{ws} cfg={cfg_id} seed={seed}: no había filas nuevas.")
                    continue

                df_all = pd.concat([df_all, df_new], ignore_index=True)
                df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

                save_seq2one_metrics(df_all, name=name, base_dir=base_dir)
                if verbose:
                    print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

                del df_new
                gc.collect()

    return df_all

In [82]:
#df_mlp_all_sizes = load_seq2one_metrics_if_exists(name="mlp", base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics")

In [83]:
MLP_ALL_TRAIN = '''
df_mlp_all_sizes = run_mlp_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="mlp",   # genera seq2one_ridge_metrics.parquet
    verbose=True,
)
'''

# **TUNEO**

## **12. Tuning de Hiperparámetros del MLP con Optuna**

Para optimizar el modelo **MLP** se adopta un enfoque de *hyperparameter tuning* utilizando **Optuna**, una biblioteca de optimización basada en búsqueda adaptativa.

**Estrategia adoptada**

Siguiendo la metodología definida en el proyecto:

**1. Durante el tuning**

- Se utiliza una **seed fija (42)** para garantizar comparabilidad entre configuraciones.
- Se optimiza exclusivamente sobre el **split VALID**.
- La métrica objetivo es **R² en VALID**.
- Se aplica **early stopping** para evitar sobreentrenamiento y reducir el costo computacional.
- El proceso de optimización se realiza en dos etapas:

  - **Tuning de arquitectura**  
    Profundidad, anchura (`hidden_dims`), `dropout`, `activation`, etc.

  - **Tuning de hiperparámetros de aprendizaje**  
    `learning rate`, `weight decay` (y eventualmente scheduler), manteniendo fija la mejor arquitectura encontrada.

**2. Después del tuning**

- Se seleccionan las mejores configuraciones (**top 1 o top 3**).
- Cada configuración se evalúa con **múltiples seeds (10–20)** para medir estabilidad.
- Se reportan las siguientes métricas agregadas:

  - **R² medio**
  - **R² desviación estándar**
  - **Gap VALID–TEST**

**Justificación del uso de Optuna**

Optuna permite:

- Explorar el espacio de hiperparámetros de forma más eficiente que una grilla tradicional.
- Reducir el número de combinaciones necesarias.
- Priorizar configuraciones prometedoras mediante búsqueda adaptativa.
- Mantener un equilibrio entre rigor metodológico y costo computacional.

Este enfoque es consistente con el pipeline actual y con la evaluación multi-seed previamente aplicada en la selección de `window_size`.

In [84]:
!pip install optuna

In [85]:
import math
import optuna

def objective_mlp_seq2one(
    trial: optuna.Trial,
    *,
    loaders: dict,
    in_dim: int,
    device: torch.device,
    # ---- entrenamiento (fijos para tuning) ----
    max_epochs: int = 30,
    patience: int = 5,
    verbose: bool = False,
) -> float:
    """
    Objective de Optuna para tunear hiperparámetros del MLP (seq2one).

    Estrategia (alineada al proyecto):
      - Seed fija = 42 durante tuning (comparabilidad entre trials)
      - Selección por VALID (métrica objetivo: R²_valid)
      - Early stopping por valid_mse (implementado en train_mlp)
      - Luego, fuera de Optuna, evaluación multi-seed sobre top configs

    Retorna:
      - R² en VALID (float), para maximizar.
    """

    # ------------------------------------------------------------
    # 0) Seed fija para el tuning
    # ------------------------------------------------------------
    set_seed(42)

    # ------------------------------------------------------------
    # 1) Espacio de búsqueda (arquitectura + aprendizaje)
    #    Nota: mantenemos el espacio discreto/estructurado para MLP
    # ------------------------------------------------------------
    activation = trial.suggest_categorical("activation", ["relu", "gelu"])
    dropout = trial.suggest_float("dropout", 0.0, 0.4)

    n_layers = trial.suggest_int("n_layers", 1, 4)
    hidden_base = trial.suggest_categorical("hidden_base", [64, 128, 256, 512])
    pattern = trial.suggest_categorical("pattern", ["flat", "decreasing"])

    if pattern == "flat":
        hidden_dims = [hidden_base] * n_layers
    else:
        # decreciente, con piso 32 para evitar capas degeneradas
        hidden_dims = []
        h = hidden_base
        for _ in range(n_layers):
            hidden_dims.append(int(max(h, 32)))
            h = h // 2

    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 3e-3, log=True)

    # ------------------------------------------------------------
    # 2) Entrenar 1 corrida (seed fija) + métricas VALID/TEST
    # ------------------------------------------------------------
    try:
        model, best_valid_mse, epochs_ran = train_mlp(
            loaders,
            in_dim=in_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            activation=activation,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            device=device,
            verbose=verbose,
        )

        metrics_valid, metrics_test = get_metrics_torch(
            bundle={"valid": {"X": loaders["valid"].dataset.tensors[0].numpy(),
                              "y": loaders["valid"].dataset.tensors[1].numpy()},
                    "test":  {"X": loaders["test"].dataset.tensors[0].numpy(),
                              "y": loaders["test"].dataset.tensors[1].numpy()},
                    # estos campos no se usan aquí por get_metrics_torch si ya lo tienes basado en bundle;
                    # si tu get_metrics_torch requiere bundle real, pásale el bundle original en vez de esto.
                   },
            model=model,
            device=device,
        )
    except RuntimeError as e:
        # típico: OOM u otro fallo numérico -> prune
        raise optuna.TrialPruned(str(e))
    except Exception as e:
        # cualquier error inesperado -> prune para continuar el estudio
        raise optuna.TrialPruned(str(e))

    # ------------------------------------------------------------
    # 3) Objective: R² en VALID
    # ------------------------------------------------------------
    r2_valid = metrics_valid.get("R2", None)
    if r2_valid is None or (isinstance(r2_valid, float) and (math.isnan(r2_valid) or math.isinf(r2_valid))):
        raise optuna.TrialPruned("R2_valid inválido o no disponible")

    r2_valid = float(r2_valid)

    # ------------------------------------------------------------
    # 4) Guardar extras del trial (útil para debug/selección top3)
    # ------------------------------------------------------------
    trial.set_user_attr("hidden_dims", hidden_dims)
    trial.set_user_attr("metrics_valid", metrics_valid)
    trial.set_user_attr("metrics_test", metrics_test)
    trial.set_user_attr("best_valid_mse", float(best_valid_mse))
    trial.set_user_attr("epochs_ran", int(epochs_ran))

    return r2_valid

Antes de ejecutar `study.optimize(...)` conviene dejar 3 cosas establecidas para que Optuna corra estable y para que el resultado sea reutilizable.

**1) Preparar los loaders una sola vez**

Crear `bundle` y `loaders` (`train/valid/test`) **fuera** del `objective` y pasarlos al `objective`.  
Esto evita recomputar ventanas/datasets en cada *trial*.

**2) Definir el “setup” del estudio**

- **Dirección**: `maximize` (métrica objetivo: **R²_valid**).
- **Sampler**: `TPESampler(seed=42)` para reproducibilidad del proceso de búsqueda  
  (además de la seed fija del entrenamiento dentro del `objective`).
- *(Opcional)* **Pruner** para cortar trials malos y reducir costo computacional.

**3) Decidir cómo va a guardar resultados**

Guardar el `study` (y/o un DF con `study.trials_dataframe()`) para:
- recuperar el **top 3**,
- y correr evaluación **multi-seed** después.

In [86]:
# =========================
# 0) Preparar BUNDLE + LOADERS (una sola vez)
# =========================

# Elegir el window_size y target del tuning
L = 180
target = "delta_60"

# Crear bundle 2D (MLP: flatten_X=True)
(bundle,) = create_bundles(
    window_size=L,
    targets=[target],
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    flatten_X=True,   # <- clave para MLP (entrada 2D)
)

# Crear DataLoaders
loaders = make_loaders_from_bundle(
    bundle,
    batch_size=16384,
    num_workers=0,     # recomendado para reproducibilidad durante tuning
)

in_dim = L * 36  # 36 features si ese es tu n_features real
print("OK loaders:", {k: len(v.dataset) for k, v in loaders.items()})
print("in_dim:", in_dim)

H60 Train: (327972, 6480) (327972,)
H60 Valid: (70228, 6480) (70228,)
H60 Test : (70590, 6480) (70590,)
Scaler H60: StandardScaler
OK loaders: {'train': 327972, 'valid': 70228, 'test': 70590}
in_dim: 6480


### **12.1. Coarse Tuning**

#### **Aplicación**

In [88]:
# ============================================================
# TUNEO GRUESO (COARSE) MLP con Optuna (SEQ2ONE)
# - Menos epochs + espacio recortado => más rápido
# - Persistente (SQLite en Drive) + snapshot parquet por trial
# - Objetivo: MAXIMIZAR R² en VALID (seed fija=42)
# ============================================================

import time
import math
import warnings
from pathlib import Path

import optuna
import torch

# ------------------------------------------------------------
# 0) Silenciar logs/warnings molestos (opcional)
# ------------------------------------------------------------
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1) Objective COARSE (usa: set_seed + train_mlp + bundle/get_metrics)
# ------------------------------------------------------------
def objective_mlp_coarse(
    trial: optuna.Trial,
    *,
    loaders: dict,
    bundle: dict,
    in_dim: int,
    device: torch.device,
    max_epochs: int = 15,
    patience: int = 3,
    verbose: bool = False,
) -> float:
    # Seed fija durante tuning
    set_seed(42)

    # ---- Espacio recortado (arquitectura + aprendizaje) ----
    activation = trial.suggest_categorical("activation", ["relu", "gelu"])
    dropout = trial.suggest_float("dropout", 0.0, 0.30)

    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_base = trial.suggest_categorical("hidden_base", [64, 128, 256, 512])
    pattern = trial.suggest_categorical("pattern", ["flat", "decreasing"])

    if pattern == "flat":
        hidden_dims = [hidden_base] * n_layers
    else:
        hidden_dims = []
        h = hidden_base
        for _ in range(n_layers):
            hidden_dims.append(int(max(h, 32)))
            h = h // 2

    lr = trial.suggest_float("lr", 2e-4, 2e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 3e-4, log=True)

    # ---- Entrenamiento + métricas ----
    try:
        model, best_valid_mse, epochs_ran = train_mlp(
            loaders,
            in_dim=in_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            activation=activation,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            device=device,
            verbose=verbose,
        )

        metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)

    except RuntimeError as e:
        # OOM u otros fallos => prune
        raise optuna.TrialPruned(str(e))
    except Exception as e:
        raise optuna.TrialPruned(str(e))

    r2_valid = float(metrics_valid.get("R2", float("nan")))
    if math.isnan(r2_valid) or math.isinf(r2_valid):
        raise optuna.TrialPruned("R2_valid inválido")

    # Guardar info útil para análisis posterior
    trial.set_user_attr("hidden_dims", hidden_dims)
    trial.set_user_attr("metrics_valid", metrics_valid)
    trial.set_user_attr("metrics_test", metrics_test)
    trial.set_user_attr("best_valid_mse", float(best_valid_mse))
    trial.set_user_attr("epochs_ran", int(epochs_ran))

    return r2_valid


# ------------------------------------------------------------
# 2) Persistencia + callbacks (progreso + snapshot parquet)
# ------------------------------------------------------------
study_name = "mlp_coarse_delta60_L180"

base_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna")
base_dir.mkdir(parents=True, exist_ok=True)

db_path = base_dir / "mlp_tuning.db"
storage_url = f"sqlite:///{db_path}"

snap_path = base_dir / f"{study_name}_trials.parquet"

t0_global = time.perf_counter()

def progress_callback(study: optuna.Study, trial: optuna.Trial) -> None:
    dt_trial = None
    if trial.datetime_start is not None and trial.datetime_complete is not None:
        dt_trial = (trial.datetime_complete - trial.datetime_start).total_seconds()

    elapsed = time.perf_counter() - t0_global
    status = trial.state.name

    best_val = None
    best_num = None
    if study.best_trial is not None and study.best_value is not None:
        best_val = study.best_value
        best_num = study.best_trial.number

    msg = f"[OPTUNA-COARSE] trial={trial.number:03d} | state={status}"
    if trial.value is not None:
        msg += f" | R2_valid={trial.value:.6f}"
    if dt_trial is not None:
        msg += f" | dt={dt_trial:.1f}s"
    msg += f" | elapsed={elapsed/60:.1f}m"
    if best_val is not None:
        msg += f" | best=trial{best_num} R2={best_val:.6f}"
    print(msg)

def snapshot_callback(study: optuna.Study, trial: optuna.Trial) -> None:
    df = study.trials_dataframe()
    df.to_parquet(snap_path, index=False)


# ------------------------------------------------------------
# 3) Crear/reanudar Study COARSE y correr
# ------------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_url,
    load_if_exists=True,
    sampler=sampler,
    pruner=pruner,
)

print("Trials existentes:", len(study.trials))
print("DB:", db_path)
print("Snapshot parquet:", snap_path)

# IMPORTANT:
# - n_trials aquí es el "grueso": 20–25 suele ser suficiente
target_total = 25
remaining = max(target_total - len(study.trials), 0)

study.optimize(
    lambda t: objective_mlp_coarse(
        t,
        loaders=loaders,
        bundle=bundle,          # <-- pasar el bundle real (valid/test) para métricas
        in_dim=in_dim,
        device=device,
        max_epochs=15,
        patience=3,
        verbose=False,
    ),
    n_trials=remaining,
    callbacks=[progress_callback, snapshot_callback],
)

print("best R2_valid (COARSE):", study.best_value)
print("best params (COARSE):", study.best_params)

df_trials = study.trials_dataframe()

Trials existentes: 1
DB: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna/mlp_tuning.db
Snapshot parquet: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna/mlp_coarse_delta60_L180_trials.parquet


ValueError: Record does not exist.

#### **Observaciones de resultados**

In [89]:
df_trials

,number,value,datetime_start,datetime_complete,duration,params_dropout,params_lr,params_weight_decay,user_attrs_activation,user_attrs_best_valid_mse,user_attrs_epochs_ran,user_attrs_hidden_dims,user_attrs_metrics_test,user_attrs_metrics_valid,state
0,0,NaN,2026-03-03 22:59:47.605477,2026-03-03 22:59:47.673745,0 days 00:00:00.068268,0.169963,0.000681,0.000162,NaN,NaN,NaN,NaN,NaN,NaN,PRUNED
1,1,0.460167,2026-03-03 23:04:35.349564,2026-03-03 23:08:07.937016,0 days 00:03:32.587452,0.169963,0.000681,0.000162,relu,1226.493578,18.0,"[512, 256, 128]","{'MAE': 36.84072326426392, 'RMSE': 62.46213667...","{'MAE': 21.4316248858716, 'RMSE': 35.021329999...",COMPLETE
2,2,0.474078,2026-03-03 23:08:08.006727,2026-03-03 23:11:35.107544,0 days 00:03:27.100817,0.187893,0.000436,0.000043,relu,1194.889702,18.0,"[512, 256, 128]","{'MAE': 34.810076594410354, 'RMSE': 59.3945994...","{'MAE': 21.242863857845336, 'RMSE': 34.5671763...",COMPLETE
3,3,0.459707,2026-03-03 23:11:35.163850,2026-03-03 23:15:03.312800,0 days 00:03:28.148950,0.144647,0.000649,0.000120,relu,1227.539429,18.0,"[512, 256, 128]","{'MAE': 36.46702423855485, 'RMSE': 60.79520985...","{'MAE': 21.60389342924938, 'RMSE': 35.03625900...",COMPLETE
4,4,0.462332,2026-03-03 23:15:03.373494,2026-03-03 23:18:34.274572,0 days 00:03:30.901078,0.196646,0.000405,0.000280,relu,1221.576522,18.0,"[512, 256, 128]","{'MAE': 34.810799297987245, 'RMSE': 59.4836946...","{'MAE': 21.932972262084558, 'RMSE': 34.9510589...",COMPLETE
5,5,0.471964,2026-03-03 23:18:34.332609,2026-03-03 23:22:03.231746,0 days 00:03:28.899137,0.206595,0.000450,0.000046,relu,1199.692117,18.0,"[512, 256, 128]","{'MAE': 35.13070983244941, 'RMSE': 60.18103574...","{'MAE': 21.225245815549965, 'RMSE': 34.6365717...",COMPLETE
6,6,0.471440,2026-03-03 23:22:03.280103,2026-03-03 23:25:29.811153,0 days 00:03:26.531050,0.154672,0.000474,0.000100,relu,1200.881401,18.0,"[512, 256, 128]","{'MAE': 34.69275569124494, 'RMSE': 59.28522339...","{'MAE': 21.442204443908327, 'RMSE': 34.6537355...",COMPLETE
7,7,0.475360,2026-03-03 23:25:29.874110,2026-03-03 23:28:56.051027,0 days 00:03:26.176917,0.174556,0.000471,0.000123,relu,1191.975010,18.0,"[512, 256, 128]","{'MAE': 35.30794865162971, 'RMSE': 60.12869974...","{'MAE': 20.9991185679489, 'RMSE': 34.524990836...",COMPLETE
8,8,0.464301,2026-03-03 23:28:56.105162,2026-03-03 23:32:23.345640,0 days 00:03:27.240478,0.151160,0.000471,0.000070,relu,1217.101882,18.0,"[512, 256, 128]","{'MAE': 35.07281791904915, 'RMSE': 59.99314871...","{'MAE': 21.62420010697461, 'RMSE': 34.88698681...",COMPLETE
9,9,0.461264,2026-03-03 23:32:23.404126,2026-03-03 23:35:52.604114,0 days 00:03:29.199988,0.176486,0.000621,0.000048,relu,1224.002164,18.0,"[512, 256, 128]","{'MAE': 35.259184530666005, 'RMSE': 60.4288860...","{'MAE': 22.05222815798814, 'RMSE': 34.98574248...",COMPLETE


1. **El modelo ganador está claramente identificado**  
   El mejor resultado corresponde al **Trial 22** con un **R²_valid = 0.4799**.  
   Hiperparámetros asociados:
   - `hidden_dims = [512, 256, 128]`
   - `activation = relu`
   - `dropout ≈ 0.178`
   - `lr ≈ 5.35e-4`
   - `weight_decay ≈ 1.9e-4`

   Se trata de una arquitectura profunda (3 capas) con patrón decreciente y regularización moderada.


2. **Existe un patrón estructural muy consistente en los mejores trials**  
   Entre los trials **16, 17, 21, 22 y 24** se repite prácticamente la misma estructura:

   - `hidden_base = 512`
   - `n_layers = 3`
   - `pattern = decreasing`
   - Arquitectura efectiva: `[512, 256, 128]`
   - `activation = relu`
   - `dropout` entre **0.16 – 0.19**
   - `lr` alrededor de **5e-4**
   - `weight_decay` entre **4e-5 – 2e-4**

   Todos ellos obtienen **R²_valid ≈ 0.47 – 0.48**, lo que indica que el espacio óptimo ya fue correctamente identificado en el coarse tuning.

3. **Las arquitecturas pequeñas no son competitivas**  
   Configuraciones con:
   - 1 sola capa (`[64]`, `[128]`)
   - hidden_base bajo (64, 128)
   - o profundidad baja

   Obtuvieron R²_valid en el rango **0.34 – 0.43**.

   En este problema, el MLP necesita **capacidad estructural suficiente** para capturar relaciones no lineales en la ventana L=180.


4. **El patrón "decreasing" domina claramente sobre "flat"**  
   Las mejores configuraciones usan sistemáticamente:
   - `pattern = decreasing`
   - Estructura tipo embudo: `[512, 256, 128]`

   Las configuraciones `flat` quedaron consistentemente por debajo.  
   Esto sugiere que la compresión progresiva de representación es beneficiosa para este target.

5. **La activación ReLU supera a GELU en este setup**  
   Aunque se probaron ambas, los mejores resultados pertenecen sistemáticamente a `relu`.  
   Las variantes con `gelu` quedaron en general por debajo del óptimo.

6. **El rango óptimo de dropout es medio (~0.15 – 0.19)**  
   - Dropout muy bajo (< 0.05) tiende a menor generalización.
   - Dropout muy alto (> 0.25) degrada el desempeño.
   - El rango medio (~0.16 – 0.19) aparece repetidamente en el top.

   Esto indica que el modelo sí requiere regularización, pero no agresiva.

7. **El learning rate óptimo es moderado (~5e-4)**  
   Los mejores trials se concentran alrededor de:
   - `lr ≈ 4.5e-4 – 6e-4`

   Valores más altos o más bajos no muestran mejoras sistemáticas.

8. **El proceso coarse ya muestra convergencia estructural**  
   Mejores resultados:

   - Trial 22 → 0.4799  
   - Trial 17 → 0.4763  
   - Trial 21 → 0.4750  
   - Trial 16 → 0.4699  

   Se observa estabilización del R²_valid alrededor de **0.47 – 0.48**.  
   No aparecen mejoras significativas adicionales al seguir explorando el mismo espacio amplio.

9. **Conclusión general del coarse tuning**

   Para el target `delta_60` con `L = 180`, la configuración estructural óptima es:

   - MLP profundo (3 capas)
   - Arquitectura decreciente
   - `[512, 256, 128]`
   - `activation = relu`
   - `dropout ≈ 0.17 – 0.19`
   - `lr ≈ 5e-4`
   - `weight_decay` bajo–medio

   El **R²_valid máximo alcanzado ≈ 0.48**.

10. **Siguiente paso**

   Realizar un **fine tuning** centrado exclusivamente en esta arquitectura:

   - Fijar `hidden_dims = [512, 256, 128]`
   - Ajustar finamente:
     - `dropout` en rango estrecho (0.14 – 0.22)
     - `lr` en rango (4e-4 – 7e-4)
     - `weight_decay` en rango bajo–medio

   Luego:
   - Seleccionar **top 1 o top 3**
   - Ejecutar evaluación **multi-seed (10–20 seeds)**
   - Reportar media, desviación estándar y gap VALID–TEST para validar robustez.

### **12.2. Fine Tuning**

#### **Aplicación**

In [90]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)

def _ts():
    return time.strftime("%H:%M:%S")

def train_mlp(
    loaders: dict,
    *,
    # ---- arquitectura ----
    in_dim: int,
    hidden_dims: list[int],
    dropout: float = 0.0,
    activation: str = "relu",
    # ---- entrenamiento ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,
):
    if verbose:
        print(
            f"[{_ts()}] [TRAIN] START | in_dim={in_dim} hidden_dims={hidden_dims} "
            f"dropout={dropout} act={activation} lr={lr} wd={weight_decay} "
            f"max_epochs={max_epochs} patience={patience} device={device.type}"
        )

    t_global = time.perf_counter()

    model = MLPSeq2One(
        in_dim=in_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
        activation=activation,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0
    epochs_ran = 0

    for epoch in range(1, max_epochs + 1):
        epochs_ran = epoch
        t_epoch = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={(train_loss_sum/max(train_n,1)):.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        valid_mse = evaluate_mse(model, loaders["valid"], device)

        improved = valid_mse < (best_valid - 1e-9)
        if improved:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            dt_epoch = time.perf_counter() - t_epoch
            print(
                f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag} | dt_epoch={dt_epoch:.2f}s"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | best_valid_mse={best_valid:.6f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} | epochs_ran={epochs_ran} | dt_total={dt_all:.2f}s")

    return model, best_valid, epochs_ran

In [91]:
# ============================================================
# TUNEO FINO (FINE) MLP con Optuna (SEQ2ONE)
# - Arquitectura fija (según coarse): [512, 256, 128] + relu
# - Ajuste fino de dropout / lr / weight_decay
# - Persistente (SQLite en Drive) + snapshot parquet por trial
# - Objetivo: MAXIMIZAR R² en VALID (seed fija=42)
# ============================================================

import time
import math
import warnings
from pathlib import Path

import optuna
import torch


# ------------------------------------------------------------
# 0) Silenciar logs/warnings (opcional)
# ------------------------------------------------------------
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1) Objective FINE (usa: set_seed + train_mlp + get_metrics_torch)
#    - Fija arquitectura ganadora del coarse:
#      hidden_dims=[512,256,128], activation=relu
#    - Tunea solo: dropout, lr, weight_decay
# ------------------------------------------------------------
def objective_mlp_fine(
    trial: optuna.Trial,
    *,
    loaders: dict,
    bundle: dict,
    in_dim: int,
    device: torch.device,
    max_epochs: int = 30,
    patience: int = 5,
    verbose: bool = False,
) -> float:
    # Seed fija durante tuning
    set_seed(42)

    # ---- Arquitectura fija (según coarse) ----
    hidden_dims = [512, 256, 128]
    activation = "relu"

    # ---- Espacio fino (centrado en lo observado en coarse) ----
    # dropout óptimo observado ~0.16–0.19 (abrimos un poco)
    dropout = trial.suggest_float("dropout", 0.14, 0.22)

    # lr observado ~4.5e-4–6e-4 (abrimos un poco)
    lr = trial.suggest_float("lr", 4e-4, 7e-4, log=True)

    # weight_decay observado ~4e-5–2e-4 (abrimos un poco)
    weight_decay = trial.suggest_float("weight_decay", 3e-5, 3e-4, log=True)

    # ---- Entrenamiento + métricas ----
    try:
        model, best_valid_mse, epochs_ran = train_mlp(
            loaders,
            in_dim=in_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            activation=activation,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            device=device,
            verbose=verbose,
        )

        metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)

    except RuntimeError as e:
        raise optuna.TrialPruned(str(e))
    except Exception as e:
        raise optuna.TrialPruned(str(e))

    r2_valid = float(metrics_valid.get("R2", float("nan")))
    if math.isnan(r2_valid) or math.isinf(r2_valid):
        raise optuna.TrialPruned("R2_valid inválido")

    # Guardar info útil para análisis posterior
    trial.set_user_attr("hidden_dims", hidden_dims)
    trial.set_user_attr("activation", activation)
    trial.set_user_attr("metrics_valid", metrics_valid)
    trial.set_user_attr("metrics_test", metrics_test)
    trial.set_user_attr("best_valid_mse", float(best_valid_mse))
    trial.set_user_attr("epochs_ran", int(epochs_ran))

    return r2_valid


# ------------------------------------------------------------
# 2) Persistencia + callbacks (progreso + snapshot parquet)
# ------------------------------------------------------------
study_name = "mlp_fine_delta60_L180"

base_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna")
base_dir.mkdir(parents=True, exist_ok=True)

db_path = base_dir / "mlp_tuning.db"
storage_url = f"sqlite:///{db_path}"

snap_path = base_dir / f"{study_name}_trials.parquet"

t0_global = time.perf_counter()

def progress_callback(study: optuna.Study, trial: optuna.Trial) -> None:
    dt_trial = None
    if trial.datetime_start is not None and trial.datetime_complete is not None:
        dt_trial = (trial.datetime_complete - trial.datetime_start).total_seconds()

    elapsed = time.perf_counter() - t0_global
    status = trial.state.name

    # ---- OJO: en algunos Optuna, best_trial/best_value lanza ValueError si no hay COMPLETE ----
    best_val = None
    best_num = None
    try:
        best_val = study.best_value
        best_num = study.best_trial.number
    except ValueError:
        # aún no existe ningún trial COMPLETE
        pass

    msg = f"[OPTUNA-FINE] trial={trial.number:03d} | state={status}"
    if trial.value is not None:
        msg += f" | R2_valid={trial.value:.6f}"
    if dt_trial is not None:
        msg += f" | dt={dt_trial:.1f}s"
    msg += f" | elapsed={elapsed/60:.1f}m"
    if best_val is not None:
        msg += f" | best=trial{best_num} R2={best_val:.6f}"
    print(msg)

def snapshot_callback(study: optuna.Study, trial: optuna.Trial) -> None:
    df = study.trials_dataframe()
    df.to_parquet(snap_path, index=False)


# ------------------------------------------------------------
# 3) Crear/reanudar Study FINE y correr
# ------------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_url,
    load_if_exists=True,
    sampler=sampler,
    pruner=pruner,
)

print("Trials existentes:", len(study.trials))
print("DB:", db_path)
print("Snapshot parquet:", snap_path)

# Recomendación: 15–25 trials para fine suele ser suficiente
target_total = 25
remaining = max(target_total - len(study.trials), 0)

study.optimize(
    lambda t: objective_mlp_fine(
        t,
        loaders=loaders,
        bundle=bundle,          # <-- pasar el bundle real (valid/test) para métricas
        in_dim=in_dim,
        device=device,
        max_epochs=30,
        patience=5,
        verbose=False,
    ),
    n_trials=remaining,
    callbacks=[progress_callback, snapshot_callback],
)

try:
    print("best R2_valid (FINE):", study.best_value)
    print("best params (FINE):", study.best_params)
except ValueError:
    print("No hay trials COMPLETE aún (todos PRUNED/FAIL). Revisar rango de búsqueda o errores de entrenamiento.")

df_trials = study.trials_dataframe()
# df_trials.sort_values("value", ascending=False).head(10)

Trials existentes: 25
DB: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna/mlp_tuning.db
Snapshot parquet: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna/mlp_fine_delta60_L180_trials.parquet
best R2_valid (FINE): 0.4781030466592304
best params (FINE): {'dropout': 0.1874469384923283, 'lr': 0.0005134159265022657, 'weight_decay': 6.599341496128554e-05}


#### **Observaciones de resultados**

In [92]:
df_trials

,number,value,datetime_start,datetime_complete,duration,params_dropout,params_lr,params_weight_decay,user_attrs_activation,user_attrs_best_valid_mse,user_attrs_epochs_ran,user_attrs_hidden_dims,user_attrs_metrics_test,user_attrs_metrics_valid,state
0,0,NaN,2026-03-03 22:59:47.605477,2026-03-03 22:59:47.673745,0 days 00:00:00.068268,0.169963,0.000681,0.000162,NaN,NaN,NaN,NaN,NaN,NaN,PRUNED
1,1,0.460167,2026-03-03 23:04:35.349564,2026-03-03 23:08:07.937016,0 days 00:03:32.587452,0.169963,0.000681,0.000162,relu,1226.493578,18.0,"[512, 256, 128]","{'MAE': 36.84072326426392, 'RMSE': 62.46213667...","{'MAE': 21.4316248858716, 'RMSE': 35.021329999...",COMPLETE
2,2,0.474078,2026-03-03 23:08:08.006727,2026-03-03 23:11:35.107544,0 days 00:03:27.100817,0.187893,0.000436,0.000043,relu,1194.889702,18.0,"[512, 256, 128]","{'MAE': 34.810076594410354, 'RMSE': 59.3945994...","{'MAE': 21.242863857845336, 'RMSE': 34.5671763...",COMPLETE
3,3,0.459707,2026-03-03 23:11:35.163850,2026-03-03 23:15:03.312800,0 days 00:03:28.148950,0.144647,0.000649,0.000120,relu,1227.539429,18.0,"[512, 256, 128]","{'MAE': 36.46702423855485, 'RMSE': 60.79520985...","{'MAE': 21.60389342924938, 'RMSE': 35.03625900...",COMPLETE
4,4,0.462332,2026-03-03 23:15:03.373494,2026-03-03 23:18:34.274572,0 days 00:03:30.901078,0.196646,0.000405,0.000280,relu,1221.576522,18.0,"[512, 256, 128]","{'MAE': 34.810799297987245, 'RMSE': 59.4836946...","{'MAE': 21.932972262084558, 'RMSE': 34.9510589...",COMPLETE
5,5,0.471964,2026-03-03 23:18:34.332609,2026-03-03 23:22:03.231746,0 days 00:03:28.899137,0.206595,0.000450,0.000046,relu,1199.692117,18.0,"[512, 256, 128]","{'MAE': 35.13070983244941, 'RMSE': 60.18103574...","{'MAE': 21.225245815549965, 'RMSE': 34.6365717...",COMPLETE
6,6,0.471440,2026-03-03 23:22:03.280103,2026-03-03 23:25:29.811153,0 days 00:03:26.531050,0.154672,0.000474,0.000100,relu,1200.881401,18.0,"[512, 256, 128]","{'MAE': 34.69275569124494, 'RMSE': 59.28522339...","{'MAE': 21.442204443908327, 'RMSE': 34.6537355...",COMPLETE
7,7,0.475360,2026-03-03 23:25:29.874110,2026-03-03 23:28:56.051027,0 days 00:03:26.176917,0.174556,0.000471,0.000123,relu,1191.975010,18.0,"[512, 256, 128]","{'MAE': 35.30794865162971, 'RMSE': 60.12869974...","{'MAE': 20.9991185679489, 'RMSE': 34.524990836...",COMPLETE
8,8,0.464301,2026-03-03 23:28:56.105162,2026-03-03 23:32:23.345640,0 days 00:03:27.240478,0.151160,0.000471,0.000070,relu,1217.101882,18.0,"[512, 256, 128]","{'MAE': 35.07281791904915, 'RMSE': 59.99314871...","{'MAE': 21.62420010697461, 'RMSE': 34.88698681...",COMPLETE
9,9,0.461264,2026-03-03 23:32:23.404126,2026-03-03 23:35:52.604114,0 days 00:03:29.199988,0.176486,0.000621,0.000048,relu,1224.002164,18.0,"[512, 256, 128]","{'MAE': 35.259184530666005, 'RMSE': 60.4288860...","{'MAE': 22.05222815798814, 'RMSE': 34.98574248...",COMPLETE


1. **Mejor resultado y magnitud de mejora frente al coarse**  
   El mejor trial del fine tuning corresponde al **Trial 12** con un **R²_valid = 0.4781**.  
   En comparación con los mejores resultados observados en el coarse tuning (≈ 0.47), la mejora es **moderada pero consistente** (≈ +0.008).  
   Esto confirma que la arquitectura identificada en el coarse (`[512, 256, 128]`) era adecuada y que el fine tuning está ajustando principalmente hiperparámetros de optimización.

2. **El rendimiento máximo se concentra en un rango estrecho**  
   Los mejores trials se agrupan en un intervalo relativamente pequeño de desempeño:

   - Trial 12: 0.4781  
   - Trial 7: 0.4754  
   - Trial 2: 0.4741  
   - Trial 20: 0.4733  
   - Trial 24: 0.4729  

   Esto sugiere que el modelo alcanza un **plateau de rendimiento alrededor de R² ≈ 0.47–0.48**, y que mejoras adicionales mediante tuning fino probablemente serán marginales.

3. **Dropout óptimo en un rango medio (≈ 0.17–0.19)**  
   Los mejores trials tienden a concentrarse aproximadamente entre **0.17 y 0.19**:

   - Trial 12: 0.187  
   - Trial 7: 0.175  
   - Trial 2: 0.188  
   - Trial 20: 0.196  

   Valores más bajos (~0.14–0.15) o más altos (>0.21) siguen funcionando, pero no aparecen entre los mejores resultados.  
   Esto sugiere que el MLP requiere **regularización moderada** para estabilizar el entrenamiento.

4. **Learning rate óptimo en un rango intermedio (~4e-4 – 6e-4)**  
   Los mejores trials utilizan learning rates cercanos a:

   - Trial 2: 4.36e-4  
   - Trial 7: 4.71e-4  
   - Trial 12: 5.13e-4  
   - Trial 20: 5.56e-4  

   Esto confirma que el rango elegido para fine tuning fue adecuado y que el modelo converge mejor con **tasas de aprendizaje moderadas**, evitando extremos demasiado bajos o demasiado altos.

5. **Weight decay óptimo relativamente bajo**  
   Los mejores resultados se obtienen con valores pequeños de regularización L2:

   - Trial 2: 4.3e-5  
   - Trial 12: 6.6e-5  
   - Trial 20: 3.2e-5  
   - Trial 24: 5.7e-5  

   Esto indica que el modelo **no necesita regularización fuerte** y que valores moderadamente bajos son suficientes para controlar el sobreajuste.

6. **Early stopping consistente entre trials**  
   Todos los trials completos entrenaron aproximadamente **18 epochs** antes de detenerse, lo que sugiere que:

   - El criterio de **patience = 5** está funcionando correctamente.
   - El modelo converge de forma estable en ese rango de epochs.
   - No se observan entrenamientos excesivamente largos ni inestables.

7. **Métricas VALID coherentes con el incremento en R²**  
   En los mejores trials se observa:

   - **MAE_valid ≈ 21.0 – 21.3**
   - **RMSE_valid ≈ 34.4 – 34.7**

   Esto es consistente con el aumento en R²_valid, indicando que el fine tuning mejora ligeramente la capacidad del modelo para explicar la varianza sin cambios drásticos en el error absoluto.

8. **Comportamiento en TEST estable entre configuraciones**  
   Las métricas de test en los mejores trials se mantienen relativamente cercanas:

   - **MAE_test ≈ 34.6 – 35.3**
   - **RMSE_test ≈ 59.2 – 60.4**

   Esto sugiere que las mejoras observadas en VALID **no están acompañadas de un deterioro evidente en TEST**, aunque la validación definitiva requiere análisis multi-seed.

9. **Conclusión general del fine tuning**  
   Para el target `delta_60` con `L = 180`, la configuración final del MLP queda caracterizada por:

   - Arquitectura: `[512, 256, 128]`
   - Activación: `relu`
   - Dropout óptimo ≈ **0.17–0.19**
   - Learning rate ≈ **4e-4 – 6e-4**
   - Weight decay ≈ **3e-5 – 7e-5**

   El rendimiento máximo alcanzado se sitúa en torno a **R²_valid ≈ 0.47–0.48**.

10. **Siguiente paso recomendado**  
   Seleccionar las **3 mejores configuraciones** del fine tuning (por ejemplo Trials 12, 7 y 2) y ejecutar evaluación **multi-seed (10–20 seeds)** para medir:

   - R²_valid medio  
   - R²_valid desviación estándar  
   - R²_test medio  
   - Gap VALID–TEST  

   La configuración final debería elegirse maximizando el **R² promedio**, minimizando la **variabilidad entre seeds**, y manteniendo controlado el **gap de generalización**.

## **13. Evaluación multi-seed**

### **13.1. Extraer Top-3 configuraciones (por R²_valid)**

In [93]:
df_trials.sort_values("value", ascending=False).head(3)

,number,value,datetime_start,datetime_complete,duration,params_dropout,params_lr,params_weight_decay,user_attrs_activation,user_attrs_best_valid_mse,user_attrs_epochs_ran,user_attrs_hidden_dims,user_attrs_metrics_test,user_attrs_metrics_valid,state
12,12,0.478103,2026-03-03 23:42:49.754222,2026-03-03 23:46:18.276515,0 days 00:03:28.522293,0.187447,0.000513,0.000066,relu,1185.743834,18.0,"[512, 256, 128]","{'MAE': 34.66232446187529, 'RMSE': 59.27856461...","{'MAE': 21.00799934131227, 'RMSE': 34.43463188...",COMPLETE
7,7,0.475360,2026-03-03 23:25:29.874110,2026-03-03 23:28:56.051027,0 days 00:03:26.176917,0.174556,0.000471,0.000123,relu,1191.975010,18.0,"[512, 256, 128]","{'MAE': 35.30794865162971, 'RMSE': 60.12869974...","{'MAE': 20.9991185679489, 'RMSE': 34.524990836...",COMPLETE
2,2,0.474078,2026-03-03 23:08:08.006727,2026-03-03 23:11:35.107544,0 days 00:03:27.100817,0.187893,0.000436,0.000043,relu,1194.889702,18.0,"[512, 256, 128]","{'MAE': 34.810076594410354, 'RMSE': 59.3945994...","{'MAE': 21.242863857845336, 'RMSE': 34.5671763...",COMPLETE


In [95]:
top_k = 3

# 1) Filtrar trials válidos (sin NaN) y ordenar por value desc
df_top = (
    df_trials
    .dropna(subset=["value"])
    .sort_values("value", ascending=False)
    .head(top_k)
    .reset_index(drop=True)
)

# 2) Extraer lista de dicts con params
top_params_list = df_top[
    ["params_dropout", "params_lr", "params_weight_decay"]
].to_dict(orient="records")

# 3) Mostrar resumen compacto
cols_show = [
    "number",
    "value",
    "params_dropout",
    "params_lr",
    "params_weight_decay",
    "user_attrs_epochs_ran",
]

display(df_top[cols_show])

top_params_list

,number,value,params_dropout,params_lr,params_weight_decay,user_attrs_epochs_ran
0,12,0.478103,0.187447,0.000513,0.000066,18.0
1,7,0.475360,0.174556,0.000471,0.000123,18.0
2,2,0.474078,0.187893,0.000436,0.000043,18.0


[{'params_dropout': 0.1874469384923283,
  'params_lr': 0.0005134159265022657,
  'params_weight_decay': 6.599341496128554e-05},
 {'params_dropout': 0.17455560149136928,
  'params_lr': 0.0004708035764380365,
  'params_weight_decay': 0.00012273661723331348},
 {'params_dropout': 0.18789267873576293,
  'params_lr': 0.0004364941798491282,
  'params_weight_decay': 4.296509486733455e-05}]

### **13.2. Evaluación multi-seed (20 seeds) sobre Top-3**

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path

def _ts() -> str:
    return time.strftime("%H:%M:%S")

# -------------------------
# Seeds (20)
# -------------------------
seeds = [
    1, 42, 123, 777, 9090, 2024, 2121, 31415, 3333, 4444,
    5555, 6666, 8888, 9999, 1010, 1111, 1212, 1313, 1414, 1515
]

# -------------------------
# Arquitectura fija (según coarse/fine)
# -------------------------
ARCH_FIXED = {
    "hidden_dims": [512, 256, 128],
    "activation": "relu",
}

# -------------------------
# Parámetros fijos del setup
# -------------------------
L = 180
n_features = 36
in_dim = L * n_features

# -------------------------
# Persistencia incremental (reanudable)
# -------------------------
out_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/multiseed")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "mlp_fine_top3_multiseed.parquet"

if out_path.exists():
    df_hist = pd.read_parquet(out_path)
else:
    df_hist = pd.DataFrame()

rows = []

print(f"[{_ts()}] [MS] out_path={out_path}")
print(f"[{_ts()}] [MS] hist_rows={len(df_hist)}")

# -------------------------
# Loop TOP-3 configs
# top_params_list debe tener dicts con:
#   {'params_dropout':..., 'params_lr':..., 'params_weight_decay':...}
# o bien {'dropout':..., 'lr':..., 'weight_decay':...}
# -------------------------
for cfg_id, params in enumerate(top_params_list, start=1):

    dropout = float(params.get("params_dropout", params.get("dropout")))
    lr = float(params.get("params_lr", params.get("lr")))
    weight_decay = float(params.get("params_weight_decay", params.get("weight_decay")))

    for seed in seeds:
        # ---- Skip si ya existe (reanudar) ----
        if not df_hist.empty:
            mask = (df_hist["cfg_id"] == cfg_id) & (df_hist["seed"] == seed)
            if mask.any():
                print(f"[{_ts()}] [MS] SKIP cfg={cfg_id} seed={seed} (ya existe)")
                continue

        print(
            f"\n[{_ts()}] [MS] RUN cfg={cfg_id}/{len(top_params_list)} seed={seed} "
            f"| do={dropout:.6f} lr={lr:.6g} wd={weight_decay:.6g}"
        )

        model = None
        try:
            set_seed(seed)

            model, best_valid_mse, epochs_ran = train_mlp(
                loaders,
                in_dim=in_dim,
                hidden_dims=list(ARCH_FIXED["hidden_dims"]),
                dropout=float(dropout),
                activation=str(ARCH_FIXED["activation"]),
                lr=float(lr),
                weight_decay=float(weight_decay),
                max_epochs=30,
                patience=5,
                device=device,
                verbose=False,
            )

            m_valid, m_test = get_metrics_torch(bundle, model, device=device)

            row = {
                "cfg_id": cfg_id,
                "seed": seed,

                "R2_valid": float(m_valid.get("R2", np.nan)),
                "R2_test":  float(m_test.get("R2", np.nan)),

                "DA_valid": float(m_valid.get("DA", np.nan)),
                "DA_test":  float(m_test.get("DA", np.nan)),

                "MAE_valid": float(m_valid.get("MAE", np.nan)),
                "MAE_test":  float(m_test.get("MAE", np.nan)),

                "RMSE_valid": float(m_valid.get("RMSE", np.nan)),
                "RMSE_test":  float(m_test.get("RMSE", np.nan)),

                # tracking entrenamiento
                "best_valid_mse": float(best_valid_mse),
                "epochs_ran": int(epochs_ran),

                # HPs (fine)
                "hp_dropout": dropout,
                "hp_lr": lr,
                "hp_weight_decay": weight_decay,

                # arquitectura fija
                "hp_hidden_dims": str(ARCH_FIXED["hidden_dims"]),
                "hp_activation": ARCH_FIXED["activation"],
                "hp_in_dim": int(in_dim),
                "hp_L": int(L),
            }

            rows.append(row)

        finally:
            # limpieza para evitar leaks
            try:
                del model
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ---- Guardado incremental por corrida ----
        if rows:
            df_new = pd.DataFrame(rows)
            if df_hist.empty:
                df_hist = df_new.copy()
            else:
                df_hist = pd.concat([df_hist, df_new], ignore_index=True)

            df_hist.to_parquet(out_path, index=False)
            rows = []  # vaciar buffer
            print(f"[{_ts()}] [MS] SAVE rows={len(df_hist)} -> {out_path}")

# DF final en memoria
df_ms = df_hist.sort_values(["cfg_id", "seed"]).reset_index(drop=True)
df_ms.head()

[01:59:11] [MS] out_path=/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/multiseed/mlp_fine_top3_multiseed.parquet
[01:59:11] [MS] hist_rows=0

[01:59:11] [MS] RUN cfg=1/3 seed=1 | do=0.187447 lr=0.000513416 wd=6.59934e-05
[02:01:55] [MS] SAVE rows=1 -> /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/multiseed/mlp_fine_top3_multiseed.parquet

[02:01:55] [MS] RUN cfg=1/3 seed=42 | do=0.187447 lr=0.000513416 wd=6.59934e-05


In [ ]:
df_ms

,cfg_id,seed,R2_valid,R2_test,DA_valid,DA_test,MAE_valid,MAE_test,RMSE_valid,RMSE_test,best_epoch,epochs_ran,hp_dropout,hp_lr,hp_weight_decay,hp_d_model,hp_nhead,hp_num_layers,hp_dim_ff,hp_pooling
0,1,1,0.573644,0.464430,0.788927,0.782130,18.576141,30.963385,31.123558,60.467742,5,10,0.044647,0.000736,0.000011,64,8,2,512,last
1,1,42,0.605222,0.480187,0.802500,0.793819,17.510620,30.185562,29.948799,59.571560,6,11,0.044647,0.000736,0.000011,64,8,2,512,last
2,1,123,0.586653,0.428581,0.792427,0.789018,17.784295,29.798635,30.645075,62.458707,6,11,0.044647,0.000736,0.000011,64,8,2,512,last
3,1,777,0.577405,0.484636,0.794228,0.793435,18.140495,29.990172,30.985980,59.316096,5,10,0.044647,0.000736,0.000011,64,8,2,512,last
4,1,1010,0.584692,0.485205,0.785884,0.787428,18.284053,30.366097,30.717681,59.283338,7,12,0.044647,0.000736,0.000011,64,8,2,512,last
5,1,1111,0.573565,0.449026,0.786112,0.771109,18.551610,31.832726,31.126445,61.331159,6,11,0.044647,0.000736,0.000011,64,8,2,512,last
6,1,1212,0.576310,0.469740,0.783998,0.773921,18.759110,32.086341,31.026110,60.167205,4,9,0.044647,0.000736,0.000011,64,8,2,512,last
7,1,1313,0.589247,0.475603,0.784312,0.779829,18.204304,31.081063,30.548762,59.833690,5,10,0.044647,0.000736,0.000011,64,8,2,512,last
8,1,1414,0.585474,0.476688,0.788013,0.770100,17.959191,31.648697,30.688719,59.771724,8,13,0.044647,0.000736,0.000011,64,8,2,512,last
9,1,1515,0.580160,0.476681,0.777868,0.784275,18.721752,30.031243,30.884814,59.772140,4,9,0.044647,0.000736,0.000011,64,8,2,512,last


1. **Las tres configuraciones son competitivas, pero con diferencias claras en TEST**  
   En VALID, las tres configs suelen moverse en un rango parecido (aprox. 0.56–0.61).  
   La diferencia relevante aparece en TEST: ahí se ve mejor qué configuración generaliza mejor.

2. **cfg_id = 2 muestra el mejor desempeño típico en TEST**  
   La cfg_id=2 (dropout≈0.0847, lr≈2.46e-4, wd≈2.2e-5) alcanza varios de los mejores R²_test (incluyendo valores >0.50) y, en general, mantiene TEST más alto de forma más consistente que cfg_id=1.  
   También mantiene DA_test en torno a ~0.78–0.80 en muchas seeds, lo cual es coherente con una mejora real de generalización.

3. **cfg_id = 1 tiende a ser más “agresiva” y menos estable en TEST**  
   La cfg_id=1 (dropout≈0.0446, lr≈7.36e-4, wd≈1.1e-5) usa un learning rate bastante más alto.  
   En TEST aparecen caídas más marcadas en algunas seeds (por ejemplo valores ~0.40–0.43), lo que sugiere mayor sensibilidad a inicialización y/o mayor riesgo de sobreajuste relativo.

4. **cfg_id = 3 es buena, pero en promedio parece por debajo de cfg_id=2 en TEST**  
   La cfg_id=3 (dropout≈0.0879, lr≈2.53e-4, wd≈2e-6) tiene resultados sólidos y algunos picos en TEST (>0.50), pero también muestra seeds con TEST más bajos (por ejemplo ~0.42).  
   En conjunto, se ve competitiva, aunque no destaca tanto como cfg_id=2.

5. **El gap VALID–TEST existe en todas (normal), pero cfg_id=2 parece controlarlo mejor**  
   En las tres configs el R²_test queda por debajo del R²_valid (esperable).  
   Aun así, cfg_id=2 parece sostener mejor el rendimiento en TEST para un nivel de VALID similar, lo que es exactamente lo que buscamos en la selección final.

6. **Recomendación práctica de selección final**  
   Si su criterio principal es **generalización (R²_test alto y estabilidad por seed)**, la candidata natural para “finalista” es **cfg_id=2**.  
   Como segunda opción razonable, **cfg_id=3**.  
   cfg_id=1 quedaría como alternativa si se prioriza un poco más VALID y se tolera mayor variabilidad.

### **13.4. Resumen final para elegir la mejor config (R²_mean alto + R²_std bajo + gap controlado)**

In [ ]:
import numpy as np

# ============================================================
# 1) Agrupar resultados multi-seed por configuración (cfg_id)
#    Cada cfg_id tiene 20 seeds.
#    Aquí calculamos métricas promedio y dispersión.
# ============================================================

summary = (
    df_ms
        # Agrupar todas las filas por configuración
        .groupby("cfg_id")

        # Calcular estadísticas agregadas por grupo
        .agg(
            # --- VALID ---
            R2_valid_mean=("R2_valid", "mean"),  # promedio R2 en VALID
            R2_valid_std =("R2_valid", "std"),   # dispersión (sensibilidad a seed)

            # --- TEST ---
            R2_test_mean =("R2_test", "mean"),   # promedio R2 en TEST (generalización)
            R2_test_std  =("R2_test", "std"),    # estabilidad en TEST

            # --- Directional Accuracy ---
            DA_valid_mean=("DA_valid", "mean"),
            DA_test_mean =("DA_test", "mean"),

            # --- Error absoluto ---
            MAE_valid_mean=("MAE_valid", "mean"),
            MAE_test_mean =("MAE_test", "mean"),
        )

        # Volver a convertir cfg_id en columna normal
        .reset_index()
)

# ============================================================
# 2) Calcular GAP promedio entre VALID y TEST
#    GAP = R2_valid - R2_test
#    Si es grande → posible sobreajuste.
#    Si es pequeño → mejor transferencia a test.
# ============================================================

gap = (
    df_ms
        # Crear columna temporal con el gap por seed
        .assign(gap_valid_minus_test=lambda d: d["R2_valid"] - d["R2_test"])

        # Agrupar por configuración
        .groupby("cfg_id")["gap_valid_minus_test"]

        # Promedio del gap entre las 20 seeds
        .mean()

        # Convertir en DataFrame
        .reset_index(name="gap_valid_minus_test_mean")
)

# ============================================================
# 3) Unir métricas agregadas + gap en un solo DataFrame
# ============================================================

summary = summary.merge(gap, on="cfg_id", how="left")


# ============================================================
# 4) Ordenar según criterio de selección final
#
# Prioridad:
#   1) Mayor R2_test_mean  → mejor generalización
#   2) Menor R2_test_std   → más estabilidad
#   3) Menor gap           → menos sobreajuste
# ============================================================

summary_sorted = summary.sort_values(
    ["R2_test_mean", "R2_test_std", "gap_valid_minus_test_mean"],
    ascending=[False, True, True],  # False = maximizar, True = minimizar
).reset_index(drop=True)

summary_sorted

,cfg_id,R2_valid_mean,R2_valid_std,R2_test_mean,R2_test_std,DA_valid_mean,DA_test_mean,MAE_valid_mean,MAE_test_mean,gap_valid_minus_test_mean
0,2,0.586100,0.009491,0.476808,0.021605,0.791358,0.784349,18.050566,30.365389,0.109291
1,3,0.586208,0.008550,0.475199,0.020677,0.791879,0.783489,18.020130,30.380457,0.111010
2,1,0.582239,0.009785,0.461825,0.025825,0.787274,0.781453,18.306551,30.866381,0.120414


## **14. Conclusión práctica**

1. **La configuración 2 presenta el mejor equilibrio entre rendimiento y estabilidad**

   En promedio, alcanza el mayor `R2_test_mean` y mantiene una dispersión controlada entre seeds, lo que indica que su desempeño no depende de una inicialización particular. Esto es una señal clara de robustez estructural.

2. **Generaliza mejor que las alternativas evaluadas**

   Comparada con las configuraciones 1 y 3, la configuración 2 sostiene de manera más consistente el rendimiento en TEST, con un gap VALID–TEST controlado y sin evidencia de sobreajuste significativo.

3. **Selección final de hiperparámetros**

   Se adopta formalmente la siguiente configuración:

   - `dropout ≈ 0.0847`
   - `lr ≈ 2.46e-4`
   - `weight_decay ≈ 2.2e-5`
   - Arquitectura fija:
     - `d_model = 64`
     - `nhead = 8`
     - `num_layers = 2`
     - `dim_ff = 512`
     - `pooling = "last"`

4. **Cierre metodológico**

   La selección se basa en un proceso completo y formal:

   - Coarse tuning para exploración estructural.
   - Fine tuning para ajuste fino de hiperparámetros.
   - Selección de top-3 configuraciones.
   - Evaluación multi-seed (20 seeds).
   - Análisis de medias, dispersión y gap VALID–TEST.

   En consecuencia, la configuración 2 puede considerarse la versión final robusta del Transformer seq2one para `delta_60` con `L=180`.